# Interactive: a single gravel river responds to its inputs

Drag the sliders to change the river's **water input**, **sediment input**, and **base level**, and watch the equilibrium long profile respond — steeper with more sediment, gentler with more water, and translated up or down with base level (the essence of *Lane's balance*).

This runs GRLP itself, live in your browser (via Pyodide) — no install, no server.

In [ ]:
%pip install -q grlp networkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

import grlp


def equilibrium_profile(Q_water, Q_sed, z_bl):
    """Single-segment equilibrium long profile for the given inputs."""
    lp = grlp.LongProfile()
    lp.basic_constants()
    lp.bedload_lumped_constants()
    lp.set_hydrologic_constants()
    lp.set_x(dx=1000., nx=60, x0=1000.)
    lp.set_z(S0=-1e-2, z1=z_bl)
    lp.set_Q(Q_water)
    lp.set_B(100.)
    lp.set_niter(3)
    lp.set_uplift_rate(0.)
    lp.set_z_bl(z_bl)
    lp.set_Qs_input_upstream(Q_sed)
    lp.evolve_threshold_width_river(nt=10, dt=1e13)   # to steady state
    return lp


def draw(Q_water, Q_sed, z_bl):
    lp = equilibrium_profile(Q_water, Q_sed, z_bl)
    plt.figure(figsize=(8, 4))
    plt.plot(lp.x / 1000., lp.z, '-', lw=3)
    plt.axhline(z_bl, color='0.6', ls='--', lw=1, label='base level')
    plt.xlim(lp.x.min() / 1000., lp.x.max() / 1000.)
    plt.ylim(-50, 1300)                      # fixed axes so the shape is comparable
    plt.xlabel('Downstream distance [km]')
    plt.ylabel('Elevation [m]')
    plt.title('water Q = %.0f m$^3$/s    sediment Q$_s$ = %.3f m$^3$/s    '
              'base level = %.0f m' % (Q_water, Q_sed, z_bl))
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
widgets.interact(
    draw,
    Q_water=widgets.FloatSlider(value=100, min=20, max=600, step=20,
                                description='Water Q'),
    Q_sed=widgets.FloatSlider(value=0.02, min=0.005, max=0.06, step=0.005,
                              description='Sediment Qs', readout_format='.3f'),
    z_bl=widgets.FloatSlider(value=0, min=-40, max=40, step=5,
                             description='Base level'),
);